# EX — Machine Learning Fundamentals Real-World Exercises

Covers train/test split, a linear regression baseline, a logistic regression classifier,
and proper evaluation metrics — using scikit-learn (`pip install scikit-learn --break-system-packages` if needed).


In [ ]:
import numpy as np
np.random.seed(0)

# Synthetic churn dataset: features -> will_churn (0/1)
n = 500
tenure_months = np.random.uniform(1, 60, n)
monthly_spend = np.random.uniform(10, 200, n)
support_tickets = np.random.poisson(2, n)

# True (unknown to us in practice) relationship
logits = -0.05*tenure_months + 0.01*monthly_spend + 0.4*support_tickets - 2
prob_churn = 1 / (1 + np.exp(-logits))
churn = (np.random.rand(n) < prob_churn).astype(int)

X = np.stack([tenure_months, monthly_spend, support_tickets], axis=1)
y = churn
print("Churn rate:", y.mean())


## 1. Train/Test Split — never skip this
**Pointer:** always evaluate on data the model didn't see during training.

In [ ]:
from sklearn.model_selection import train_test_split
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=0, stratify=y)
print(X_train.shape, X_test.shape)


## 2. Baseline: Logistic Regression

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

model = LogisticRegression()
model.fit(X_train_scaled, y_train)


### TODO 1
Compute accuracy, precision, recall, and F1 on the test set (use `sklearn.metrics`). Which metric matters most for a churn model, and why might accuracy alone be misleading?

In [ ]:
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score

# TODO
preds = None
acc = prec = rec = f1 = None
print(acc, prec, rec, f1)


<details><summary>Solution + discussion</summary>

```python
preds = model.predict(X_test_scaled)
acc = accuracy_score(y_test, preds)
prec = precision_score(y_test, preds)
rec = recall_score(y_test, preds)
f1 = f1_score(y_test, preds)
```
For churn prediction, **recall** (catching actual churners) often matters more than
accuracy — missing a churner is usually costlier than a false alarm, and if churn is
rare, a model predicting "no churn" for everyone can still score high accuracy while
being useless.
</details>


## 3. A Stronger Baseline: Random Forest
**Pointer:** tree ensembles are usually the strongest baseline on tabular data, and don't require feature scaling.

In [ ]:
from sklearn.ensemble import RandomForestClassifier

rf = RandomForestClassifier(n_estimators=200, random_state=0)
rf.fit(X_train, y_train)  # no scaling needed for trees
rf_preds = rf.predict(X_test)
print("RF accuracy:", accuracy_score(y_test, rf_preds))
print("RF recall:", recall_score(y_test, rf_preds))
print("Feature importances:", dict(zip(["tenure","spend","tickets"], rf.feature_importances_)))


### TODO 2
Use `cross_val_score` (5-fold) to get a more reliable recall estimate for the random forest than a single train/test split.

In [ ]:
from sklearn.model_selection import cross_val_score

# TODO
cv_scores = None
print(cv_scores)


<details><summary>Solution</summary>

```python
cv_scores = cross_val_score(RandomForestClassifier(n_estimators=200, random_state=0), X, y, cv=5, scoring='recall')
```
</details>

## Key Takeaways
- Always split train/test (or cross-validate); never evaluate on training data.
- Start with a simple baseline (logistic regression) before a more complex model.
- Pick your evaluation metric based on the business problem, not just accuracy.
- Tree ensembles (random forest) are a strong, low-effort baseline on tabular data.
